# CS180 Project 3, Part B — Flow Matching from Scratch

This is the complete standalone solution used to produce the submitted report.
It implements the official UNet blocks, one-step denoising, time-conditioned
flow matching, class conditioning with 10% dropout, and classifier-free
guidance. Training uses all 60,000 MNIST training images; no personal images are
used in Part B.


## Runtime and reproducibility

Select a GPU runtime. Run the setup and source cells once, then run the desired
experiment groups in order. Every group checkpoints after each epoch, so an
interrupted runtime resumes rather than starting over. Seed 180 is fixed for
data order, held-out corruption, and sampling comparisons.


In [ ]:
from pathlib import Path
import sys, subprocess

assert sys.version_info >= (3, 9)
REQUIREMENTS = 'matplotlib\nnumpy\npillow\ntorch\ntorchvision\n'
Path('requirements.txt').write_text(REQUIREMENTS, encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchvision',
                'matplotlib', 'numpy'], check=True)
print('Environment ready:', sys.version)


## 1.1 UNet operations and exact architectures


In [ ]:
# The exact submitted implementation is embedded for portability.
MODEL_SOURCE = '"""Networks and flow-matching algorithms for CS180 Project 3 Part B.\n\nThe tensor shapes and layer parameters follow the official Fall 2026 diagrams.\nTime runs from noise at t=0 to data at t=1, and every flow model predicts\nvelocity x_1 - x_0.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass Conv(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),\n            nn.BatchNorm2d(out_channels),\n            nn.GELU(),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass DownConv(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),\n            nn.BatchNorm2d(out_channels),\n            nn.GELU(),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass UpConv(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.ConvTranspose2d(\n                in_channels, out_channels, kernel_size=4, stride=2, padding=1\n            ),\n            nn.BatchNorm2d(out_channels),\n            nn.GELU(),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass Flatten(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.net = nn.Sequential(nn.AvgPool2d(kernel_size=7), nn.GELU())\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass Unflatten(nn.Module):\n    def __init__(self, in_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.ConvTranspose2d(\n                in_channels, in_channels, kernel_size=7, stride=7, padding=0\n            ),\n            nn.BatchNorm2d(in_channels),\n            nn.GELU(),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            Conv(in_channels, out_channels), Conv(out_channels, out_channels)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass DownBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            DownConv(in_channels, out_channels), ConvBlock(out_channels, out_channels)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass UpBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            UpConv(in_channels, out_channels), ConvBlock(out_channels, out_channels)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass UnconditionalUNet(nn.Module):\n    def __init__(self, in_channels: int = 1, num_hiddens: int = 128):\n        super().__init__()\n        d = num_hiddens\n        self.init = ConvBlock(in_channels, d)\n        self.down1 = DownBlock(d, d)\n        self.down2 = DownBlock(d, 2 * d)\n        self.flatten = Flatten()\n        self.unflatten = Unflatten(2 * d)\n        self.up1 = UpBlock(4 * d, d)\n        self.up2 = UpBlock(2 * d, d)\n        self.final = nn.Sequential(\n            ConvBlock(2 * d, d),\n            nn.Conv2d(d, in_channels, kernel_size=3, stride=1, padding=1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."\n        x0 = self.init(x)\n        x1 = self.down1(x0)\n        x2 = self.down2(x1)\n        z = self.unflatten(self.flatten(x2))\n        u1 = self.up1(torch.cat((x2, z), dim=1))\n        u2 = self.up2(torch.cat((x1, u1), dim=1))\n        return self.final(torch.cat((x0, u2), dim=1))\n\n\nclass FCBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(in_channels, out_channels),\n            nn.GELU(),\n            nn.Linear(out_channels, out_channels),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass _ConditionalBackbone(nn.Module):\n    """Shared official UNet backbone with two conditioning injection points."""\n\n    def __init__(self, in_channels: int, num_hiddens: int):\n        super().__init__()\n        d = num_hiddens\n        self.d = d\n        self.init = ConvBlock(in_channels, d)\n        self.down1 = DownBlock(d, d)\n        self.down2 = DownBlock(d, 2 * d)\n        self.flatten = Flatten()\n        self.unflatten = Unflatten(2 * d)\n        self.up1 = UpBlock(4 * d, d)\n        self.up2 = UpBlock(2 * d, d)\n        self.final = nn.Sequential(\n            ConvBlock(2 * d, d),\n            nn.Conv2d(d, in_channels, kernel_size=3, stride=1, padding=1),\n        )\n\n    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:\n        x0 = self.init(x)\n        x1 = self.down1(x0)\n        x2 = self.down2(x1)\n        z = self.unflatten(self.flatten(x2))\n        return x0, x1, x2, z\n\n    def decode(\n        self,\n        x0: torch.Tensor,\n        x1: torch.Tensor,\n        x2: torch.Tensor,\n        z: torch.Tensor,\n        second_condition: torch.Tensor,\n    ) -> torch.Tensor:\n        u1 = self.up1(torch.cat((x2, z), dim=1))\n        u1 = u1 * second_condition\n        u2 = self.up2(torch.cat((x1, u1), dim=1))\n        return self.final(torch.cat((x0, u2), dim=1))\n\n\nclass TimeConditionalUNet(nn.Module):\n    def __init__(\n        self, in_channels: int = 1, num_classes: int = 10, num_hiddens: int = 64\n    ):\n        super().__init__()\n        del num_classes  # Kept for compatibility with the official starter signature.\n        d = num_hiddens\n        self.backbone = _ConditionalBackbone(in_channels, d)\n        self.fc1_t = FCBlock(1, 2 * d)\n        self.fc2_t = FCBlock(1, d)\n\n    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:\n        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."\n        t = t.float().reshape(-1, 1)\n        t1 = self.fc1_t(t).unsqueeze(-1).unsqueeze(-1)\n        t2 = self.fc2_t(t).unsqueeze(-1).unsqueeze(-1)\n        x0, x1, x2, z = self.backbone.encode(x)\n        z = z * t1\n        return self.backbone.decode(x0, x1, x2, z, t2)\n\n\nclass ClassConditionalUNet(nn.Module):\n    def __init__(\n        self, in_channels: int = 1, num_classes: int = 10, num_hiddens: int = 64\n    ):\n        super().__init__()\n        d = num_hiddens\n        self.num_classes = num_classes\n        self.backbone = _ConditionalBackbone(in_channels, d)\n        self.fc1_t = FCBlock(1, 2 * d)\n        self.fc2_t = FCBlock(1, d)\n        self.fc1_c = FCBlock(num_classes, 2 * d)\n        self.fc2_c = FCBlock(num_classes, d)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        c: torch.Tensor,\n        t: torch.Tensor,\n        mask: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor:\n        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."\n        t = t.float().reshape(-1, 1)\n        one_hot = F.one_hot(c.long(), self.num_classes).float()\n        if mask is not None:\n            one_hot = one_hot * mask.float().reshape(-1, 1)\n\n        t1 = self.fc1_t(t).unsqueeze(-1).unsqueeze(-1)\n        t2 = self.fc2_t(t).unsqueeze(-1).unsqueeze(-1)\n        c1 = self.fc1_c(one_hot).unsqueeze(-1).unsqueeze(-1)\n        c2 = self.fc2_c(one_hot).unsqueeze(-1).unsqueeze(-1)\n\n        x0, x1, x2, z = self.backbone.encode(x)\n        z = c1 * z + t1\n        u1 = self.backbone.up1(torch.cat((x2, z), dim=1))\n        u1 = c2 * u1 + t2\n        u2 = self.backbone.up2(torch.cat((x1, u1), dim=1))\n        return self.backbone.final(torch.cat((x0, u2), dim=1))\n\n\ndef time_fm_forward(\n    unet: TimeConditionalUNet, x_1: torch.Tensor, num_ts: int\n) -> torch.Tensor:\n    unet.train()\n    del num_ts  # The training algorithm samples continuous t ~ Uniform([0, 1]).\n    batch = x_1.shape[0]\n    x_0 = torch.randn_like(x_1)\n    t = torch.rand(batch, device=x_1.device)\n    t_image = t.reshape(-1, 1, 1, 1)\n    x_t = (1.0 - t_image) * x_0 + t_image * x_1\n    target = x_1 - x_0\n    return F.mse_loss(unet(x_t, t), target)\n\n\n@torch.inference_mode()\ndef time_fm_sample(\n    unet: TimeConditionalUNet,\n    img_wh: tuple[int, int],\n    num_ts: int,\n    seed: int = 0,\n    num_samples: int = 40,\n) -> torch.Tensor:\n    unet.eval()\n    device = next(unet.parameters()).device\n    generator = torch.Generator(device=device).manual_seed(seed)\n    x = torch.randn((num_samples, 1, *img_wh), generator=generator, device=device)\n    step = 1.0 / num_ts\n    for index in range(num_ts):\n        t = torch.full((num_samples,), index / num_ts, device=device)\n        x = x + step * unet(x, t)\n    return x\n\n\ndef class_fm_forward(\n    unet: ClassConditionalUNet,\n    x_1: torch.Tensor,\n    c: torch.Tensor,\n    p_uncond: float,\n    num_ts: int,\n) -> torch.Tensor:\n    unet.train()\n    del num_ts  # num_ts controls Euler sampling, not the training distribution.\n    batch = x_1.shape[0]\n    x_0 = torch.randn_like(x_1)\n    t = torch.rand(batch, device=x_1.device)\n    t_image = t.reshape(-1, 1, 1, 1)\n    x_t = (1.0 - t_image) * x_0 + t_image * x_1\n    target = x_1 - x_0\n    keep_mask = (torch.rand(batch, device=x_1.device) >= p_uncond).float()\n    return F.mse_loss(unet(x_t, c, t, keep_mask), target)\n\n\n@torch.inference_mode()\ndef class_fm_sample(\n    unet: ClassConditionalUNet,\n    c: torch.Tensor,\n    img_wh: tuple[int, int],\n    num_ts: int,\n    guidance_scale: float = 5.0,\n    seed: int = 0,\n    return_cache: bool = False,\n) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:\n    unet.eval()\n    device = next(unet.parameters()).device\n    c = c.to(device)\n    batch = len(c)\n    generator = torch.Generator(device=device).manual_seed(seed)\n    x = torch.randn((batch, 1, *img_wh), generator=generator, device=device)\n    step = 1.0 / num_ts\n    cache = []\n    keep = torch.ones(batch, device=device)\n    drop = torch.zeros(batch, device=device)\n    cache_every = max(1, num_ts // 10)\n    for index in range(num_ts):\n        t = torch.full((batch,), index / num_ts, device=device)\n        u_uncond = unet(x, c, t, drop)\n        u_cond = unet(x, c, t, keep)\n        velocity = u_uncond + guidance_scale * (u_cond - u_uncond)\n        x = x + step * velocity\n        if return_cache and (index % cache_every == 0 or index == num_ts - 1):\n            cache.append(x.detach().cpu())\n    if return_cache:\n        return x, torch.stack(cache, dim=1)\n    return x\n\n\nclass TimeConditionalFM(nn.Module):\n    def __init__(self, unet: TimeConditionalUNet, num_ts: int = 50):\n        super().__init__()\n        self.unet = unet\n        self.num_ts = num_ts\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return time_fm_forward(self.unet, x, self.num_ts)\n\n    def sample(self, img_wh: tuple[int, int], seed: int = 0) -> torch.Tensor:\n        return time_fm_sample(self.unet, img_wh, self.num_ts, seed)\n\n\nclass ClassConditionalFM(nn.Module):\n    def __init__(\n        self,\n        unet: ClassConditionalUNet,\n        num_ts: int = 300,\n        p_uncond: float = 0.1,\n    ):\n        super().__init__()\n        self.unet = unet\n        self.num_ts = num_ts\n        self.p_uncond = p_uncond\n\n    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:\n        return class_fm_forward(self.unet, x, c, self.p_uncond, self.num_ts)\n\n    def sample(\n        self,\n        c: torch.Tensor,\n        img_wh: tuple[int, int] = (28, 28),\n        guidance_scale: float = 5.0,\n        seed: int = 0,\n    ) -> torch.Tensor:\n        return class_fm_sample(\n            self.unet, c, img_wh, self.num_ts, guidance_scale, seed\n        )\n\n\ndef architecture_smoke_test() -> dict[str, tuple[int, ...]]:\n    x = torch.randn(2, 1, 28, 28)\n    t = torch.tensor([0.25, 0.75])\n    c = torch.tensor([2, 7])\n    outputs = {\n        "unconditional": tuple(UnconditionalUNet(1, 8)(x).shape),\n        "time": tuple(TimeConditionalUNet(1, 10, 8)(x, t).shape),\n        "class": tuple(ClassConditionalUNet(1, 10, 8)(x, c, t).shape),\n    }\n    expected = (2, 1, 28, 28)\n    if any(shape != expected for shape in outputs.values()):\n        raise AssertionError(outputs)\n    return outputs\n\n\nif __name__ == "__main__":\n    print(architecture_smoke_test())\n'
Path('partb_models.py').write_text(MODEL_SOURCE, encoding='utf-8')
exec(compile(MODEL_SOURCE, 'partb_models.py', 'exec'), globals())
print('Architecture smoke test:', architecture_smoke_test())


## Training and visualization driver

The driver uses fresh noise for every fetched batch, Adam and the handout
learning rates, exponential decay after each flow-matching epoch, continuous
`t ~ Uniform([0,1])` during training, and Euler integration during sampling.
The local 2 GB GPU path uses gradient accumulation only for the D=128 denoiser,
preserving the required effective batch size of 256.


In [ ]:
RUNNER_SOURCE = '"""Run every required CS180 Project 3 Part B experiment.\n\nExamples\n--------\npython run_partb.py --group denoise\npython run_partb.py --group time\npython run_partb.py --group class\npython run_partb.py --group no-scheduler\n\nEach group checkpoints after every epoch, so an interrupted run can be resumed.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom torchvision.datasets import MNIST\nfrom torchvision.transforms import ToTensor\nfrom torchvision.utils import make_grid\n\nfrom partb_models import (\n    ClassConditionalFM,\n    ClassConditionalUNet,\n    TimeConditionalFM,\n    TimeConditionalUNet,\n    UnconditionalUNet,\n    architecture_smoke_test,\n)\n\n\nSEED = 180\nSIGMAS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef seed_everything(seed: int = SEED) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef make_loader(images: torch.Tensor, labels: torch.Tensor, batch_size: int) -> DataLoader:\n    dataset = TensorDataset(images[:, None].float() / 255.0, labels)\n    generator = torch.Generator().manual_seed(SEED)\n    return DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=True,\n        num_workers=0,\n        pin_memory=DEVICE.type == "cuda",\n        generator=generator,\n    )\n\n\ndef save_grid(\n    images: torch.Tensor,\n    path: Path,\n    title: str,\n    nrow: int,\n    normalize: bool = False,\n) -> None:\n    images = images.detach().float().cpu()\n    if normalize:\n        lo = images.amin(dim=(1, 2, 3), keepdim=True)\n        hi = images.amax(dim=(1, 2, 3), keepdim=True)\n        images = (images - lo) / (hi - lo).clamp_min(1e-8)\n    grid = make_grid(images.clamp(0, 1), nrow=nrow, padding=2, pad_value=0.15)\n    rows = math.ceil(len(images) / nrow)\n    fig, ax = plt.subplots(figsize=(max(7, nrow * 1.05), max(2.5, rows * 1.15)))\n    ax.imshow(grid.permute(1, 2, 0), interpolation="nearest")\n    ax.set_title(title)\n    ax.axis("off")\n    fig.tight_layout()\n    fig.savefig(path, dpi=150, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef save_loss(losses: list[float], path: Path, title: str) -> None:\n    array = np.asarray(losses, dtype=np.float64)\n    window = min(100, len(array))\n    smooth = np.convolve(array, np.ones(window) / window, mode="valid")\n    fig, ax = plt.subplots(figsize=(8.5, 3.5))\n    ax.plot(array, color="#73a9c2", alpha=0.28, linewidth=0.55, label="step")\n    ax.plot(\n        np.arange(window - 1, len(array)),\n        smooth,\n        color="#c46d39",\n        linewidth=1.7,\n        label=f"{window}-step mean",\n    )\n    ax.set(xlabel="Optimizer step", ylabel="Mean squared error", title=title)\n    ax.grid(alpha=0.2)\n    ax.legend(frameon=False)\n    fig.tight_layout()\n    fig.savefig(path, dpi=160)\n    plt.close(fig)\n\n\ndef checkpoint_path(out: Path, name: str, epoch: int) -> Path:\n    return out / "checkpoints" / f"{name}_epoch{epoch}.pt"\n\n\ndef latest_checkpoint(out: Path, name: str, max_epoch: int) -> tuple[int, Path] | None:\n    for epoch in range(max_epoch, 0, -1):\n        path = checkpoint_path(out, name, epoch)\n        if path.is_file():\n            return epoch, path\n    return None\n\n\ndef load_resume(\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    out: Path,\n    name: str,\n    epochs: int,\n    scheduler: torch.optim.lr_scheduler.LRScheduler | None = None,\n) -> tuple[int, list[float]]:\n    found = latest_checkpoint(out, name, epochs)\n    if not found:\n        return 0, []\n    epoch, path = found\n    state = torch.load(path, map_location=DEVICE, weights_only=False)\n    model.load_state_dict(state["model"])\n    optimizer.load_state_dict(state["optimizer"])\n    if scheduler is not None and state.get("scheduler") is not None:\n        scheduler.load_state_dict(state["scheduler"])\n    print(f"Resuming {name} after epoch {epoch}: {path}")\n    return epoch, list(state.get("losses", []))\n\n\ndef save_checkpoint(\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    out: Path,\n    name: str,\n    epoch: int,\n    losses: list[float],\n    scheduler: torch.optim.lr_scheduler.LRScheduler | None = None,\n) -> None:\n    path = checkpoint_path(out, name, epoch)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "epoch": epoch,\n            "model": model.state_dict(),\n            "optimizer": optimizer.state_dict(),\n            "scheduler": scheduler.state_dict() if scheduler is not None else None,\n            "losses": losses,\n        },\n        path,\n    )\n\n\ndef train_loop(\n    name: str,\n    model: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    loader: DataLoader,\n    epochs: int,\n    out: Path,\n    step_fn,\n    scheduler: torch.optim.lr_scheduler.LRScheduler | None = None,\n    epoch_callback=None,\n    accumulation_steps: int = 1,\n) -> tuple[list[float], list[float], float]:\n    start_epoch, losses = load_resume(model, optimizer, out, name, epochs, scheduler)\n    epoch_losses = []\n    started = time.perf_counter()\n    for epoch in range(start_epoch, epochs):\n        # Sampling callbacks switch the module to eval mode.  Restore training\n        # mode explicitly at every epoch so BatchNorm keeps updating.\n        model.train()\n        running = 0.0\n        count = 0\n        epoch_start = time.perf_counter()\n        optimizer.zero_grad(set_to_none=True)\n        for batch_index, (images, labels) in enumerate(loader):\n            images = images.to(DEVICE, non_blocking=True).to(\n                memory_format=torch.channels_last\n            )\n            labels = labels.to(DEVICE, non_blocking=True)\n            loss = step_fn(model, images, labels)\n            (loss / accumulation_steps).backward()\n            if (batch_index + 1) % accumulation_steps == 0 or batch_index + 1 == len(loader):\n                optimizer.step()\n                optimizer.zero_grad(set_to_none=True)\n            value = float(loss.detach())\n            losses.append(value)\n            running += value\n            count += 1\n        if scheduler is not None:\n            scheduler.step()\n        mean_loss = running / count\n        epoch_losses.append(mean_loss)\n        elapsed = time.perf_counter() - epoch_start\n        lr = optimizer.param_groups[0]["lr"]\n        print(\n            f"{name}: epoch {epoch + 1}/{epochs}, loss={mean_loss:.6f}, "\n            f"lr={lr:.6g}, {elapsed:.1f}s",\n            flush=True,\n        )\n        save_checkpoint(model, optimizer, out, name, epoch + 1, losses, scheduler)\n        if epoch_callback is not None:\n            epoch_callback(epoch + 1, model)\n    return losses, epoch_losses, time.perf_counter() - started\n\n\ndef run_denoising(\n    out: Path,\n    train_images: torch.Tensor,\n    train_labels: torch.Tensor,\n    test_images: torch.Tensor,\n) -> dict:\n    group = out / "denoise"\n    group.mkdir(parents=True, exist_ok=True)\n    micro_batch = 64 if DEVICE.type == "cuda" else 128\n    accumulation_steps = 256 // micro_batch\n    loader = make_loader(train_images, train_labels, batch_size=micro_batch)\n\n    sample = test_images[0:1, None].float() / 255.0\n    noise_generator = torch.Generator().manual_seed(SEED)\n    epsilon = torch.randn(sample.shape, generator=noise_generator)\n    noise_levels = torch.cat([sample + sigma * epsilon for sigma in SIGMAS])\n    save_grid(\n        noise_levels,\n        group / "noise_levels.png",\n        "Forward noising process",\n        nrow=7,\n    )\n\n    fixed_test = (test_images[:20, None].float() / 255.0).to(DEVICE)\n    fixed_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 1)\n    fixed_eps = torch.randn(fixed_test.shape, generator=fixed_generator, device=DEVICE)\n\n    model = UnconditionalUNet(1, 128).to(DEVICE, memory_format=torch.channels_last)\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)\n    criterion = nn.MSELoss()\n\n    def sigma_step(net, images, _labels):\n        noisy = images + 0.5 * torch.randn_like(images)\n        return criterion(net(noisy), images)\n\n    @torch.inference_mode()\n    def sigma_callback(epoch, net):\n        if epoch in (1, 5):\n            net.eval()\n            noisy = fixed_test + 0.5 * fixed_eps\n            denoised = net(noisy.to(memory_format=torch.channels_last))\n            paired = torch.stack((noisy, denoised), dim=1).flatten(0, 1)\n            save_grid(\n                paired,\n                group / f"denoise_epoch{epoch}.png",\n                f"Sigma=0.5: noisy input / denoised output after epoch {epoch}",\n                nrow=10,\n            )\n\n    losses, epoch_losses, elapsed = train_loop(\n        "denoise_sigma05",\n        model,\n        optimizer,\n        loader,\n        5,\n        group,\n        sigma_step,\n        epoch_callback=sigma_callback,\n        accumulation_steps=accumulation_steps,\n    )\n    save_loss(losses, group / "denoise_loss.png", "Single-step denoiser, sigma=0.5")\n\n    model.eval()\n    with torch.inference_mode():\n        fixed_image = fixed_test[:1]\n        fixed_noise = fixed_eps[:1]\n        ood_inputs = torch.cat([fixed_image + sigma * fixed_noise for sigma in SIGMAS])\n        ood_outputs = model(ood_inputs.to(memory_format=torch.channels_last))\n        interleaved = torch.stack((ood_inputs, ood_outputs), dim=1).flatten(0, 1)\n        save_grid(\n            interleaved,\n            group / "ood_noise_levels.png",\n            "OOD noise levels: input / denoised output",\n            nrow=7,\n        )\n\n    pure_model = UnconditionalUNet(1, 128).to(DEVICE, memory_format=torch.channels_last)\n    pure_optimizer = torch.optim.Adam(pure_model.parameters(), lr=1e-4)\n    pure_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 2)\n    pure_fixed = torch.randn(\n        fixed_test.shape, generator=pure_generator, device=DEVICE\n    )\n\n    def pure_step(net, images, _labels):\n        source = torch.randn_like(images)\n        return criterion(net(source), images)\n\n    @torch.inference_mode()\n    def pure_callback(epoch, net):\n        if epoch in (1, 5):\n            net.eval()\n            generated = net(pure_fixed.to(memory_format=torch.channels_last))\n            save_grid(\n                generated,\n                group / f"pure_noise_epoch{epoch}.png",\n                f"Single-step predictions from independent noise, epoch {epoch}",\n                nrow=10,\n            )\n\n    pure_losses, pure_epoch_losses, pure_elapsed = train_loop(\n        "denoise_pure_noise",\n        pure_model,\n        pure_optimizer,\n        loader,\n        5,\n        group,\n        pure_step,\n        epoch_callback=pure_callback,\n        accumulation_steps=accumulation_steps,\n    )\n    save_loss(\n        pure_losses,\n        group / "pure_noise_loss.png",\n        "Single-step denoiser trained from independent noise",\n    )\n    return {\n        "sigma_epoch_losses": epoch_losses,\n        "pure_epoch_losses": pure_epoch_losses,\n        "seconds": elapsed + pure_elapsed,\n        "micro_batch": micro_batch,\n        "gradient_accumulation_steps": accumulation_steps,\n        "effective_batch_size": micro_batch * accumulation_steps,\n        "interpretation": (\n            "Because the source noise is independent of the target digit, MSE rewards "\n            "the conditional mean of the training distribution. The output therefore "\n            "collapses toward a soft MNIST centroid rather than diverse digits."\n        ),\n    }\n\n\ndef run_time_conditioned(\n    out: Path, train_images: torch.Tensor, train_labels: torch.Tensor\n) -> dict:\n    group = out / "time"\n    group.mkdir(parents=True, exist_ok=True)\n    loader = make_loader(train_images, train_labels, batch_size=64)\n    model = TimeConditionalFM(TimeConditionalUNet(1, 10, 64), num_ts=50)\n    model.to(DEVICE, memory_format=torch.channels_last)\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)\n    scheduler = torch.optim.lr_scheduler.ExponentialLR(\n        optimizer, gamma=0.1 ** (1.0 / 10)\n    )\n\n    def step(net, images, _labels):\n        return net(images)\n\n    @torch.inference_mode()\n    def callback(epoch, net):\n        if epoch in (1, 5, 10):\n            samples = net.sample((28, 28), seed=SEED)\n            save_grid(\n                samples,\n                group / f"time_epoch{epoch}.png",\n                f"Time-conditioned samples, epoch {epoch}",\n                nrow=10,\n            )\n\n    losses, epoch_losses, elapsed = train_loop(\n        "time_conditioned",\n        model,\n        optimizer,\n        loader,\n        10,\n        group,\n        step,\n        scheduler=scheduler,\n        epoch_callback=callback,\n    )\n    save_loss(losses, group / "time_loss.png", "Time-conditioned flow matching")\n    return {"epoch_losses": epoch_losses, "seconds": elapsed, "num_ts": 50}\n\n\ndef run_class_conditioned(\n    out: Path,\n    train_images: torch.Tensor,\n    train_labels: torch.Tensor,\n    no_scheduler: bool,\n) -> dict:\n    group_name = "no_scheduler" if no_scheduler else "class"\n    group = out / group_name\n    group.mkdir(parents=True, exist_ok=True)\n    loader = make_loader(train_images, train_labels, batch_size=64)\n    model = ClassConditionalFM(\n        ClassConditionalUNet(1, 10, 64), num_ts=300, p_uncond=0.1\n    )\n    model.to(DEVICE, memory_format=torch.channels_last)\n    epochs = 15 if no_scheduler else 10\n    learning_rate = 1e-3 if no_scheduler else 1e-2\n    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)\n    scheduler = None\n    if not no_scheduler:\n        scheduler = torch.optim.lr_scheduler.ExponentialLR(\n            optimizer, gamma=0.1 ** (1.0 / 10)\n        )\n\n    def step(net, images, labels):\n        return net(images, labels)\n\n    labels = torch.arange(10).repeat_interleave(4)\n    milestones = (1, 5, 10, 15) if no_scheduler else (1, 5, 10)\n\n    @torch.inference_mode()\n    def callback(epoch, net):\n        if epoch in milestones:\n            samples = net.sample(labels, guidance_scale=5.0, seed=SEED)\n            save_grid(\n                samples,\n                group / f"{group_name}_epoch{epoch}.png",\n                f"Class-conditioned CFG samples, epoch {epoch}",\n                nrow=4,\n            )\n\n    losses, epoch_losses, elapsed = train_loop(\n        group_name,\n        model,\n        optimizer,\n        loader,\n        epochs,\n        group,\n        step,\n        scheduler=scheduler,\n        epoch_callback=callback,\n    )\n    save_loss(\n        losses,\n        group / f"{group_name}_loss.png",\n        "Class-conditioned flow matching"\n        + (" without a scheduler" if no_scheduler else ""),\n    )\n    result = {\n        "epoch_losses": epoch_losses,\n        "seconds": elapsed,\n        "num_ts": 300,\n        "guidance_scale": 5.0,\n        "p_uncond": 0.1,\n        "epochs": epochs,\n        "learning_rate": learning_rate,\n        "scheduler": None\n        if no_scheduler\n        else {"type": "ExponentialLR", "gamma": 0.1 ** (1.0 / 10)},\n    }\n    if no_scheduler:\n        result["compensation"] = (\n            "Replaced the decaying 1e-2 schedule with a conservative constant "\n            "1e-3 learning rate and extended training from 10 to 15 epochs."\n        )\n    return result\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--group",\n        choices=("download", "denoise", "time", "class", "no-scheduler", "all"),\n        default="all",\n    )\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--out", type=Path, default=Path("results/partb"))\n    parser.add_argument("--threads", type=int, default=min(8, os.cpu_count() or 4))\n    args = parser.parse_args()\n\n    seed_everything()\n    torch.set_num_threads(args.threads)\n    torch.set_num_interop_threads(1)\n    torch.backends.mkldnn.enabled = True\n    torch.backends.cudnn.benchmark = True\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n    args.out.mkdir(parents=True, exist_ok=True)\n    print("Architecture smoke test:", architecture_smoke_test())\n    device_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"\n    print(f"torch={torch.__version__}, device={DEVICE}, name={device_name}, threads={args.threads}")\n\n    train = MNIST(args.data, train=True, download=True, transform=ToTensor())\n    test = MNIST(args.data, train=False, download=True, transform=ToTensor())\n    if args.group == "download":\n        print(f"MNIST ready: train={len(train)}, test={len(test)}")\n        return\n\n    metrics_path = args.out / "metrics.json"\n    metrics = (\n        json.loads(metrics_path.read_text(encoding="utf-8"))\n        if metrics_path.is_file()\n        else {}\n    )\n    metrics.update(\n        {\n            "seed": SEED,\n            "torch": torch.__version__,\n            "device": str(DEVICE),\n            "device_name": device_name,\n            "train_examples": len(train),\n            "test_examples": len(test),\n            "architecture": "official Fall 2026 UNet diagram",\n        }\n    )\n\n    groups = (\n        ("denoise", "time", "class", "no-scheduler")\n        if args.group == "all"\n        else (args.group,)\n    )\n    for group in groups:\n        if group == "denoise":\n            metrics[group] = run_denoising(\n                args.out, train.data, train.targets, test.data\n            )\n        elif group == "time":\n            metrics[group] = run_time_conditioned(\n                args.out, train.data, train.targets\n            )\n        elif group == "class":\n            metrics[group] = run_class_conditioned(\n                args.out, train.data, train.targets, no_scheduler=False\n            )\n        elif group == "no-scheduler":\n            metrics[group] = run_class_conditioned(\n                args.out, train.data, train.targets, no_scheduler=True\n            )\n        metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")\n    print(json.dumps(metrics, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'
Path('run_partb.py').write_text(RUNNER_SOURCE, encoding='utf-8')
print('Wrote the resumable training driver.')


## 1.2 Single-step denoising, OOD testing, and pure-noise failure


In [ ]:
# Produces: noising visualization, epoch 1/5 denoising grids,
# full loss curve, OOD sigma sweep, pure-noise epoch 1/5 grids and explanation.
subprocess.run([sys.executable, '-u', 'run_partb.py', '--group', 'denoise'], check=True)


## 2.1–2.3 Time-conditioned flow matching


In [ ]:
# D=64, batch=64, Adam 1e-2, ExpLR, 10 epochs, 50 Euler steps.
# Saves the whole loss curve and samples after epochs 1, 5, and 10.
subprocess.run([sys.executable, '-u', 'run_partb.py', '--group', 'time'], check=True)


## 2.4–2.6 Class-conditioned flow matching with CFG


In [ ]:
# D=64, p_uncond=.1, 10 epochs, gamma=5, 300 Euler steps.
# Saves four examples of each digit after epochs 1, 5, and 10.
subprocess.run([sys.executable, '-u', 'run_partb.py', '--group', 'class'], check=True)


## Bells & Whistles — no-scheduler ablation


In [ ]:
# Independent constant-lr comparison: lr=1e-3 for 15 epochs.
subprocess.run([sys.executable, '-u', 'run_partb.py', '--group', 'no-scheduler'], check=True)


## Export

All figures, checkpoints, and `metrics.json` are written below
`results/partb/`. Download that directory and place it beside the report before
running the report asset-sync step.
